In [2]:
####  charger les données

import pandas as pd
import json

with open('../data/raw/planches.json', 'r', encoding='utf-8') as f:
    planches = json.load(f)

In [3]:
df = pd.DataFrame(planches)
print("Avant nettoyage:")
print(df['prix'].head())

# Enlever "€" et espaces, convertir en float
df['prix'] = df['prix'].str.replace('€', '').str.strip().astype(float)

print("\nAprès nettoyage:")
print(df['prix'].head())
###print(df['prix'].dtype)

Avant nettoyage:
0    431.9 €
1    774.9 €
2    479.9 €
3    323.9 €
4    448.9 €
Name: prix, dtype: object

Après nettoyage:
0    431.9
1    774.9
2    479.9
3    323.9
4    448.9
Name: prix, dtype: float64
float64


In [9]:
### Nettoyer nb_derives

print("Valeurs uniques et leurs comptes:")
print(df['nb_derives'].value_counts(dropna=False))


print("Avant nettoyage:")
print(df['nb_derives'].head())
print(df['nb_derives'].dtype)



# 1. Remplacer "2+1 dérives" par "2"
df['nb_derives'] = df['nb_derives'].str.replace('2+1 dérives', '2')

# 2. Remplacer "Non disponible" par "3"
df['nb_derives'] = df['nb_derives'].str.replace('Non disponible', '3')

# 3. Extraire le nombre et convertir en int
df['nb_derives'] = df['nb_derives'].str.extract(r'(\d+)')[0].astype(int)

print("Après nettoyage:")
print(df['nb_derives'].value_counts().sort_index())
print(df['nb_derives'].dtype)

Valeurs uniques et leurs comptes:
nb_derives
3 (Thruster)      68
5 (Tri-Quad)      21
4 (Quad)           9
2+1 dérives        6
2 (Twin)           4
Non disponible     2
1 (Single)         2
Name: count, dtype: int64
Avant nettoyage:
0    3 (Thruster)
1    3 (Thruster)
2        4 (Quad)
3    3 (Thruster)
4    3 (Thruster)
Name: nb_derives, dtype: object
object
Après nettoyage:
nb_derives
1     2
2    10
3    70
4     9
5    21
Name: count, dtype: int64
int64


In [14]:
### Nettoyer colonne Conception



print("Avant nettoyage:")
print(df['conception'].head(10))
print(df['conception'].value_counts())



# Traduire en français
df['conception'] = df['conception'].str.replace('Soft', 'Mousse')
df['conception'] = df['conception'].str.replace('Epoxy', 'Époxy')

# Normaliser les espaces autour du "/"
df['conception'] = df['conception'].str.replace(r'\s*/\s*', '/', regex=True)

print("Après nettoyage:")
print(df['conception'].value_counts())

Avant nettoyage:
0          Mousse
1          Rigide
2    Mousse/Époxy
3          Mousse
4          Mousse
5          Mousse
6          Mousse
7          Mousse
8          Mousse
9          Mousse
Name: conception, dtype: object
conception
Rigide          50
Mousse          44
Mousse/Époxy    18
Name: count, dtype: int64
Après nettoyage:
conception
Rigide          50
Mousse          44
Mousse/Époxy    18
Name: count, dtype: int64


In [17]:
### Nettoyage de la colonne Volume


print("Exemples de volume:")
print(df['volume'].head(20))
print("\nValeurs uniques (premiers 10):")
print(df['volume'].unique()[:10])





# 1. Remplacer virgules par points
df['volume'] = df['volume'].str.replace(',', '.')

# 2. Enlever "L" (avec ou sans espace)
df['volume'] = df['volume'].str.replace(r'\s*L', '', regex=True)

# 3. Splitter et convertir en liste de floats
df['volume'] = df['volume'].str.split('/').apply(lambda x: [float(v.strip()) for v in x])

print("Après nettoyage:")
print(df['volume'].head(10))

Exemples de volume:
0                               57 / 72 / 78L / 86 / 98
1     22.50 / 24.00 / 25.25 / 26.25 / 27.25 / 28.50 ...
2                                      34.5 / 43.5 / 53
3                          54 / 61 / 66 / 71 / 94 / 103
4                               57 / 72 / 78L / 86 / 98
5                               57 / 72 / 78L / 86 / 98
6                               57 / 72 / 78L / 86 / 98
7                                          42 / 48 / 56
8                                          42 / 48 / 56
9                                          72 / 82 / 98
10                                         72 / 82 / 98
11    27.25 / 28.50 / 29.50 / 31.00 / 32.00 / 33.50 ...
12    30 / 32 / 34.3 / 37.5 / 41 / 43.6 / 46.4 / 49....
13                                     31 / 36.1 / 39.4
14                   36 / 39 / 40L / 40.3 / 47.9 / 50.6
15                                   27.5 / 32.4 / 40.1
16                               22 / 30 / 33 / 39 / 44
17                          

In [19]:
print("Niveau le plus courant:")
niveau_courant = df[df['niveau'] != 'Non disponible']['niveau'].value_counts().idxmax()
print(f"{niveau_courant} : {df[df['niveau'] == niveau_courant].shape[0]}")

print("\nTaille_vagues la plus courant:")
vagues_courant = df[df['taille_vagues'] != 'Non disponible']['taille_vagues'].value_counts().idxmax()
print(f"{vagues_courant} : {df[df['taille_vagues'] == vagues_courant].shape[0]}")


# Remplacer "Non disponible" par les valeurs les plus fréquentes
df['niveau'] = df['niveau'].replace('Non disponible', 'Intermédiaire - Confirmé')
df['taille_vagues'] = df['taille_vagues'].replace('Non disponible', 'Vagues moyennes')

print("Après remplacement:")
print(f"niveau - Non disponible: {(df['niveau'] == 'Non disponible').sum()}")
print(f"taille_vagues - Non disponible: {(df['taille_vagues'] == 'Non disponible').sum()}")

print("\nValeurs uniques niveau:")
print(df['niveau'].value_counts())

print("\nValeurs uniques taille_vagues:")
print(df['taille_vagues'].value_counts())

Niveau le plus courant:
Intermédiaire - Confirmé : 65

Taille_vagues la plus courant:
Vagues moyennes : 50
Après remplacement:
niveau - Non disponible: 0
taille_vagues - Non disponible: 0

Valeurs uniques niveau:
niveau
Intermédiaire - Confirmé    70
Débutant - Intermédiaire    32
Confirmé - Expert           10
Name: count, dtype: int64

Valeurs uniques taille_vagues:
taille_vagues
Vagues moyennes    57
Petites vagues     40
Grandes vagues     12
Vagues creuses      3
Name: count, dtype: int64


In [21]:
# Convertir les listes en strings JSON pour la sauvegarde


import os

# Créer le dossier s'il n'existe pas
os.makedirs('../data/cleaned', exist_ok=True)

df_export = df.copy()
df_export['volume'] = df_export['volume'].apply(lambda x: x if isinstance(x, str) else x)

# Sauvegarder
df_export.to_json('../data/cleaned/planches_cleaned.json', orient='records', force_ascii=False, indent=2)

print("✓ Données sauvegardées dans data/cleaned/planches_cleaned.json")

✓ Données sauvegardées dans data/cleaned/planches_cleaned.json
